# 09_Schedule_Simulation

In [4]:
# ============================================================
# NB_09 — SCHEDULE SIMULATION
# 09.01 — LOAD INPUTS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from graphlib import TopologicalSorter, CycleError

PROJECT_DIR = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")
OUTPUT_DIR = PROJECT_DIR / "07_Outputs"

# 1. Cargar línea base CPM
df_baseline = pd.read_csv(
    OUTPUT_DIR / "cpm_activity_schedule.csv"
)

# 2. Cargar dependencias originales
df_dependencies = pd.read_csv(
    OUTPUT_DIR / "cpm_dependencies.csv"
)

# 3. Cargar escenarios de NB_08
df_scenarios = pd.read_excel(
    OUTPUT_DIR / "Scenario_Configuration.xlsx",
    sheet_name="Scenarios"
)

# 4. Identificar nodos y relaciones
activity_ids = set(df_baseline["ID"])

predecessors = {
    activity_id: set()
    for activity_id in activity_ids
}

for row in df_dependencies.itertuples(index=False):
    predecessors[row.Successor_ID].add(row.Predecessor_ID)

# 5. Validaciones
assert df_baseline["ID"].is_unique
assert df_dependencies["Predecessor_ID"].isin(activity_ids).all()
assert df_dependencies["Successor_ID"].isin(activity_ids).all()
assert df_scenarios["Activity_ID"].isin(activity_ids).all()

try:
    execution_order = list(
        TopologicalSorter(predecessors).static_order()
    )
    network_is_dag = True

except CycleError:
    network_is_dag = False
    raise RuntimeError("The dependency network contains a cycle.")

print("INPUTS LOADED AND VALIDATED")
print(f"Activities: {len(df_baseline)}")
print(f"Dependencies: {len(df_dependencies)}")
print(f"Scenarios: {df_scenarios['Scenario_ID'].nunique()}")
print(f"Scenario impact records: {len(df_scenarios)}")
print(f"Network is DAG: {network_is_dag}")

print("\nRELATION TYPES")
display(df_dependencies["Relation_Type"].value_counts())

print("\nSCENARIOS")
display(df_scenarios)

In [7]:
BASELINE_DURATION = float(
    pd.read_csv(
        OUTPUT_DIR / "cpm_project_summary.csv"
    ).loc[0, "Project_Duration_CPM_Days"]
)

In [8]:
# ============================================================
# 09.02 — RECONSTRUCT BASELINE CPM
# ============================================================

# 1. Duraciones por actividad
durations = (
    df_baseline
    .set_index("ID")["Duration_Plan_Days"]
    .astype(float)
    .to_dict()
)

# 2. Dependencias agrupadas por actividad sucesora
dependencies_by_successor = {
    activity_id: []
    for activity_id in df_baseline["ID"]
}

for row in df_dependencies.itertuples(index=False):
    dependencies_by_successor[row.Successor_ID].append({
        "predecessor": row.Predecessor_ID,
        "relation": row.Relation_Type,
        "lag": float(row.Lag)
    })

# 3. Recorrido hacia delante: cálculo de fechas tempranas
ES = {}
EF = {}

for activity_id in execution_order:

    duration = durations[activity_id]
    candidate_starts = [0.0]

    for dependency in dependencies_by_successor[activity_id]:

        predecessor = dependency["predecessor"]
        relation = dependency["relation"]
        lag = dependency["lag"]

        if relation == "FS":
            candidate = EF[predecessor] + lag

        elif relation == "SS":
            candidate = ES[predecessor] + lag

        elif relation == "SF":
            candidate = ES[predecessor] + lag - duration

        else:
            raise ValueError(
                f"Unsupported relation type: {relation}"
            )

        candidate_starts.append(candidate)

    ES[activity_id] = max(candidate_starts)
    EF[activity_id] = ES[activity_id] + duration

# 4. Comparación con el CPM congelado de NB_07
df_validation = df_baseline[
    ["ID", "Name", "ES_Days", "EF_Days"]
].copy()

df_validation["ES_Recalculated"] = df_validation["ID"].map(ES)
df_validation["EF_Recalculated"] = df_validation["ID"].map(EF)

df_validation["ES_Error"] = (
    df_validation["ES_Recalculated"] - df_validation["ES_Days"]
)

df_validation["EF_Error"] = (
    df_validation["EF_Recalculated"] - df_validation["EF_Days"]
)

# 5. Indicadores de validación
project_duration_recalculated = max(EF.values())

max_es_error = df_validation["ES_Error"].abs().max()
max_ef_error = df_validation["EF_Error"].abs().max()

mismatches = df_validation[
    (df_validation["ES_Error"].abs() > 1e-6) |
    (df_validation["EF_Error"].abs() > 1e-6)
]

print("BASELINE CPM RECONSTRUCTION")
print(f"Recalculated duration: {project_duration_recalculated:.2f} days")
print(f"Reference duration: {BASELINE_DURATION:.2f} days")
print(f"Maximum ES difference: {max_es_error:.6f} days")
print(f"Maximum EF difference: {max_ef_error:.6f} days")
print(f"Activities with differences: {len(mismatches)}")

if len(mismatches) > 0:
    display(
        mismatches[
            [
                "ID", "Name",
                "ES_Days", "ES_Recalculated", "ES_Error",
                "EF_Days", "EF_Recalculated", "EF_Error"
            ]
        ].head(15)
    )
else:
    print("VALIDATION PASSED: recalculated dates match NB_07.")

In [9]:
# ============================================================
# 09.03 — APPLY SCENARIO IMPACTS
# ============================================================

# 1. Validar tipos de impacto
valid_impact_types = {
    "SUPPLY_DELAY",
    "ACTIVITY_DELAY"
}

assert set(df_scenarios["Impact_Type"]).issubset(valid_impact_types)

# 2. Preparar resultados
scenario_activity_results = []
scenario_supply_results = []

for scenario_id in sorted(df_scenarios["Scenario_ID"].unique()):

    scenario_rows = df_scenarios[
        df_scenarios["Scenario_ID"] == scenario_id
    ]

    # Impactos que modifican directamente la duración CPM
    activity_impacts = (
        scenario_rows[
            scenario_rows["Impact_Type"] == "ACTIVITY_DELAY"
        ]
        .groupby("Activity_ID")["Delay_Days"]
        .sum()
    )

    # Impactos de suministro: se registran por separado
    supply_impacts = scenario_rows[
        scenario_rows["Impact_Type"] == "SUPPLY_DELAY"
    ].copy()

    for row in supply_impacts.itertuples(index=False):
        scenario_supply_results.append({
            "Scenario_ID": scenario_id,
            "Activity_ID": row.Activity_ID,
            "Supply_Delay_Days": row.Delay_Days
        })

    # Crear una copia independiente de la duración de referencia
    df_scenario = df_baseline[
        ["ID", "Duration_Plan_Days"]
    ].copy()

    df_scenario["Scenario_ID"] = scenario_id

    df_scenario["Activity_Delay_Applied"] = (
        df_scenario["ID"]
        .map(activity_impacts)
        .fillna(0)
    )

    df_scenario["Duration_Simulated_Days"] = (
        df_scenario["Duration_Plan_Days"]
        + df_scenario["Activity_Delay_Applied"]
    )

    scenario_activity_results.append(df_scenario)

# 3. Consolidar resultados
df_scenario_durations = pd.concat(
    scenario_activity_results,
    ignore_index=True
)

df_scenario_supply = pd.DataFrame(
    scenario_supply_results,
    columns=[
        "Scenario_ID",
        "Activity_ID",
        "Supply_Delay_Days"
    ]
)

# 4. Validaciones
assert len(df_scenario_durations) == 135 * 3
assert df_scenario_durations.groupby("Scenario_ID")["ID"].nunique().eq(135).all()

assert (
    df_scenario_durations["Duration_Simulated_Days"]
    >= df_scenario_durations["Duration_Plan_Days"]
).all()

# Comprobar los impactos de actividad previstos
expected_activity_impacts = {
    "SC01": 0,
    "SC02": 7,
    "SC03": 7
}

actual_activity_impacts = (
    df_scenario_durations
    .groupby("Scenario_ID")["Activity_Delay_Applied"]
    .sum()
    .to_dict()
)

assert actual_activity_impacts == expected_activity_impacts

# Comprobar impactos de suministro
expected_supply_impacts = {
    "SC01": 15,
    "SC02": 0,
    "SC03": 15
}

actual_supply_impacts = (
    df_scenario_supply
    .groupby("Scenario_ID")["Supply_Delay_Days"]
    .sum()
    .to_dict()
)

for scenario_id, expected in expected_supply_impacts.items():
    assert actual_supply_impacts.get(scenario_id, 0) == expected

print("SCENARIO IMPACTS VALIDATED")
print("\nACTIVITY DURATION IMPACTS")
display(
    df_scenario_durations
    .groupby("Scenario_ID")
    .agg(
        Affected_Activities=("Activity_Delay_Applied", lambda x: (x > 0).sum()),
        Total_Activity_Delay_Days=("Activity_Delay_Applied", "sum")
    )
)

print("\nSUPPLY IMPACTS")
display(
    df_scenario_supply
    .groupby("Scenario_ID")
    .agg(
        Affected_Activities=("Activity_ID", "nunique"),
        Total_Supply_Delay_Days=("Supply_Delay_Days", "sum")
    )
)

print("\nSCENARIO DURATION DATA")
display(
    df_scenario_durations[
        df_scenario_durations["Activity_Delay_Applied"] > 0
    ]
)

In [10]:
# ============================================================
# 09.04 — RECALCULATE CPM FOR EACH SCENARIO
# ============================================================

# 1. Preparar estructura de dependencias
allowed_relations = {"FS", "SS", "FF", "SF"}

assert set(df_dependencies["Relation_Type"]).issubset(allowed_relations)

incoming = {
    activity_id: []
    for activity_id in df_baseline["ID"]
}

outgoing = {
    activity_id: []
    for activity_id in df_baseline["ID"]
}

for row in df_dependencies.itertuples(index=False):

    predecessor = row.Predecessor_ID
    successor = row.Successor_ID

    edge = {
        "Predecessor": predecessor,
        "Successor": successor,
        "Relation": row.Relation_Type,
        "Lag": float(row.Lag)
    }

    incoming[successor].append(edge)
    outgoing[predecessor].append(edge)


# 2. Función CPM
def calculate_scenario_cpm(durations, execution_order):

    durations = {
        activity_id: float(duration)
        for activity_id, duration in durations.items()
    }

    es = {}
    ef = {}

    # --------------------------------------------------------
    # FORWARD PASS
    # --------------------------------------------------------
    for activity_id in execution_order:

        duration = durations[activity_id]
        constraints = []

        for edge in incoming[activity_id]:

            predecessor = edge["Predecessor"]
            relation = edge["Relation"]
            lag = edge["Lag"]

            if relation == "FS":
                candidate = ef[predecessor] + lag

            elif relation == "SS":
                candidate = es[predecessor] + lag

            elif relation == "FF":
                candidate = ef[predecessor] + lag - duration

            elif relation == "SF":
                candidate = es[predecessor] + lag - duration

            constraints.append(candidate)

        es[activity_id] = max([0.0] + constraints)
        ef[activity_id] = es[activity_id] + duration

    project_duration = max(ef.values())

    # --------------------------------------------------------
    # BACKWARD PASS
    # --------------------------------------------------------
    ls = {}
    lf = {}

    for activity_id in reversed(execution_order):

        duration = durations[activity_id]
        constraints = []

        for edge in outgoing[activity_id]:

            successor = edge["Successor"]
            relation = edge["Relation"]
            lag = edge["Lag"]

            if relation == "FS":
                candidate = ls[successor] - lag - duration

            elif relation == "SS":
                candidate = ls[successor] - lag

            elif relation == "FF":
                candidate = lf[successor] - lag - duration

            elif relation == "SF":
                candidate = lf[successor] - lag

            constraints.append(candidate)

        if constraints:
            ls[activity_id] = min(constraints)
        else:
            ls[activity_id] = project_duration - duration

        lf[activity_id] = ls[activity_id] + duration

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------
    results = pd.DataFrame({
        "ID": list(durations.keys()),
        "Duration_Simulated_Days": list(durations.values())
    })

    results["ES"] = results["ID"].map(es)
    results["EF"] = results["ID"].map(ef)
    results["LS"] = results["ID"].map(ls)
    results["LF"] = results["ID"].map(lf)

    results["Total_Float"] = results["LS"] - results["ES"]

    results["Is_Critical"] = (
        results["Total_Float"].abs() <= 1e-6
    )

    return results, project_duration


# 3. Recalcular cada escenario
scenario_cpm_results = []
scenario_cpm_summary = []

for scenario_id in sorted(
    df_scenario_durations["Scenario_ID"].unique()
):

    scenario_data = df_scenario_durations[
        df_scenario_durations["Scenario_ID"] == scenario_id
    ]

    durations = (
        scenario_data
        .set_index("ID")["Duration_Simulated_Days"]
        .to_dict()
    )

    df_result, project_duration = calculate_scenario_cpm(
        durations,
        execution_order
    )

    df_result["Scenario_ID"] = scenario_id

    scenario_cpm_results.append(df_result)

    scenario_cpm_summary.append({
        "Scenario_ID": scenario_id,
        "Project_Duration_Days": project_duration,
        "Critical_Activities": int(
            df_result["Is_Critical"].sum()
        )
    })


# 4. Consolidar resultados
df_scenario_cpm = pd.concat(
    scenario_cpm_results,
    ignore_index=True
)

df_scenario_summary = pd.DataFrame(
    scenario_cpm_summary
)


# 5. Validar SC01: suministro sin impacto directo en CPM
sc01 = (
    df_scenario_cpm[
        df_scenario_cpm["Scenario_ID"] == "SC01"
    ]
    .set_index("ID")
    .sort_index()
)

baseline_reference = (
    df_baseline
    .set_index("ID")
    .sort_index()
)

max_es_difference = (
    sc01["ES"] - baseline_reference["ES_Days"]
).abs().max()

max_ef_difference = (
    sc01["EF"] - baseline_reference["EF_Days"]
).abs().max()

assert np.isclose(max_es_difference, 0, atol=1e-6)
assert np.isclose(max_ef_difference, 0, atol=1e-6)
assert np.isclose(
    df_scenario_summary.loc[
        df_scenario_summary["Scenario_ID"] == "SC01",
        "Project_Duration_Days"
    ].iloc[0],
    BASELINE_DURATION
)

print("SCENARIO CPM RECONSTRUCTION VALIDATED")
print(f"SC01 maximum ES difference: {max_es_difference:.6f}")
print(f"SC01 maximum EF difference: {max_ef_difference:.6f}")

print("\nPROJECT SCENARIO SUMMARY")
display(df_scenario_summary)

print("\nCRITICAL ACTIVITIES BY SCENARIO")
display(
    df_scenario_cpm[
        df_scenario_cpm["Is_Critical"]
    ][
        ["Scenario_ID", "ID", "Duration_Simulated_Days",
         "ES", "EF", "Total_Float"]
    ].sort_values(["Scenario_ID", "ID"])
)

In [11]:
# ============================================================
# 09.04.1 — DIAGNOSE SCENARIO FLOAT CALCULATION
# ============================================================

for scenario_id in ["SC01", "SC02", "SC03"]:

    df_diag = (
        df_scenario_cpm[
            df_scenario_cpm["Scenario_ID"] == scenario_id
        ]
        .copy()
        .sort_values("Total_Float")
    )

    print(f"\n========== {scenario_id} ==========")

    print("Float distribution:")
    display(
        df_diag["Total_Float"].describe()
    )

    print("\nLowest float activities:")
    display(
        df_diag[
            [
                "ID",
                "Duration_Simulated_Days",
                "ES",
                "EF",
                "LS",
                "LF",
                "Total_Float",
                "Is_Critical"
            ]
        ].head(10)
    )

    print("\nNegative floats:", (df_diag["Total_Float"] < -1e-6).sum())
    print("Zero floats:", (df_diag["Total_Float"].abs() <= 1e-6).sum())
    print("Maximum float:", df_diag["Total_Float"].max())

In [12]:
# ============================================================
# 09.04.2 — COMPARE BACKWARD PASS WITH NB_07
# ============================================================

# 1. Recuperar resultado SC01
sc01 = (
    df_scenario_cpm[
        df_scenario_cpm["Scenario_ID"] == "SC01"
    ]
    .copy()
)

# 2. Preparar referencia NB_07
reference = df_baseline[
    [
        "ID",
        "LS_Days",
        "LF_Days",
        "Total_Float_Days",
        "Is_Critical"
    ]
].copy()

# 3. Comparar
comparison = sc01.merge(
    reference,
    on="ID",
    how="inner",
    validate="one_to_one"
)

comparison["LS_Difference"] = (
    comparison["LS"] - comparison["LS_Days"]
)

comparison["LF_Difference"] = (
    comparison["LF"] - comparison["LF_Days"]
)

comparison["Float_Difference"] = (
    comparison["Total_Float"] - comparison["Total_Float_Days"]
)

print("BACKWARD PASS COMPARISON")

print(
    "Maximum LS difference:",
    comparison["LS_Difference"].abs().max()
)

print(
    "Maximum LF difference:",
    comparison["LF_Difference"].abs().max()
)

print(
    "Maximum float difference:",
    comparison["Float_Difference"].abs().max()
)

print("\nREFERENCE CRITICAL ACTIVITIES")
display(
    reference[reference["Is_Critical"]]
)

print("\nCOMPARISON OF CRITICAL CHAIN")
display(
    comparison[
        comparison["ID"].isin([12, 18, 20])
    ][
        [
            "ID",
            "LS",
            "LS_Days",
            "LS_Difference",
            "LF",
            "LF_Days",
            "LF_Difference",
            "Total_Float",
            "Total_Float_Days"
        ]
    ].sort_values("ID")
)

print("\nLARGEST LS DIFFERENCES")
display(
    comparison.assign(
        Absolute_Difference=lambda x: x["LS_Difference"].abs()
    )
    .sort_values("Absolute_Difference", ascending=False)
    .head(10)
)

# 4. Examinar dependencias de la cadena crítica
print("\nDEPENDENCIES INVOLVING CRITICAL ACTIVITIES")

display(
    df_dependencies[
        df_dependencies["Predecessor_ID"].isin([12, 18, 20])
        | df_dependencies["Successor_ID"].isin([12, 18, 20])
    ].sort_values(
        ["Predecessor_ID", "Successor_ID"]
    )
)

a está identificado el error. No está en las fórmulas FS, SS, FF o SF, sino en la condición de cierre del pase hacia atrás.

Causa: estábamos limitando la fecha de finalización del proyecto únicamente a las actividades terminales. Pero, con relaciones SF, una actividad puede tener sucesores y aun así quedar calculada con una fecha de finalización posterior al proyecto.

La corrección es imponer a todas las actividades el límite:
\[
LS_i \leq D_{\text{proyecto}}-Duración_i
\]

Y después aplicar las restricciones adicionales de sus sucesoras.
09.04.3 — Corregir el pase hacia atrás
Sustituye la función calculate_scenario_cpm por esta versión:

In [13]:
# ============================================================
# 09.04.3 — CORRECTED CPM CALCULATION
# ============================================================

def calculate_scenario_cpm(durations, execution_order):

    durations = {
        activity_id: float(duration)
        for activity_id, duration in durations.items()
    }

    es = {}
    ef = {}

    # --------------------------------------------------------
    # FORWARD PASS
    # --------------------------------------------------------
    for activity_id in execution_order:

        duration = durations[activity_id]
        constraints = []

        for edge in incoming[activity_id]:

            predecessor = edge["Predecessor"]
            relation = edge["Relation"]
            lag = edge["Lag"]

            if relation == "FS":
                candidate = ef[predecessor] + lag

            elif relation == "SS":
                candidate = es[predecessor] + lag

            elif relation == "FF":
                candidate = ef[predecessor] + lag - duration

            elif relation == "SF":
                candidate = es[predecessor] + lag - duration

            constraints.append(candidate)

        es[activity_id] = max([0.0] + constraints)
        ef[activity_id] = es[activity_id] + duration

    project_duration = max(ef.values())

    # --------------------------------------------------------
    # BACKWARD PASS
    # --------------------------------------------------------
    ls = {}
    lf = {}

    for activity_id in reversed(execution_order):

        duration = durations[activity_id]

        # Límite global: ninguna actividad puede finalizar
        # después de la fecha de finalización del proyecto.
        constraints = [
            project_duration - duration
        ]

        for edge in outgoing[activity_id]:

            successor = edge["Successor"]
            relation = edge["Relation"]
            lag = edge["Lag"]

            if relation == "FS":
                candidate = ls[successor] - lag - duration

            elif relation == "SS":
                candidate = ls[successor] - lag

            elif relation == "FF":
                candidate = lf[successor] - lag - duration

            elif relation == "SF":
                candidate = lf[successor] - lag

            constraints.append(candidate)

        ls[activity_id] = min(constraints)
        lf[activity_id] = ls[activity_id] + duration

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------
    results = pd.DataFrame({
        "ID": list(durations.keys()),
        "Duration_Simulated_Days": list(durations.values())
    })

    results["ES"] = results["ID"].map(es)
    results["EF"] = results["ID"].map(ef)
    results["LS"] = results["ID"].map(ls)
    results["LF"] = results["ID"].map(lf)

    results["Total_Float"] = results["LS"] - results["ES"]

    results["Is_Critical"] = (
        results["Total_Float"].abs() <= 1e-6
    )

    return results, project_duration


# ============================================================
# REBUILD ALL SCENARIOS
# ============================================================

scenario_cpm_results = []
scenario_cpm_summary = []

for scenario_id in sorted(
    df_scenario_durations["Scenario_ID"].unique()
):

    scenario_data = df_scenario_durations[
        df_scenario_durations["Scenario_ID"] == scenario_id
    ]

    durations = (
        scenario_data
        .set_index("ID")["Duration_Simulated_Days"]
        .to_dict()
    )

    df_result, project_duration = calculate_scenario_cpm(
        durations,
        execution_order
    )

    df_result["Scenario_ID"] = scenario_id
    scenario_cpm_results.append(df_result)

    scenario_cpm_summary.append({
        "Scenario_ID": scenario_id,
        "Project_Duration_Days": project_duration,
        "Critical_Activities": int(
            df_result["Is_Critical"].sum()
        )
    })

df_scenario_cpm = pd.concat(
    scenario_cpm_results,
    ignore_index=True
)

df_scenario_summary = pd.DataFrame(
    scenario_cpm_summary
)


# ============================================================
# VALIDATE SC01 AGAINST NB_07
# ============================================================

sc01 = (
    df_scenario_cpm[
        df_scenario_cpm["Scenario_ID"] == "SC01"
    ]
    .merge(
        df_baseline,
        on="ID",
        validate="one_to_one"
    )
)

max_es_error = (
    sc01["ES"] - sc01["ES_Days"]
).abs().max()

max_ef_error = (
    sc01["EF"] - sc01["EF_Days"]
).abs().max()

max_ls_error = (
    sc01["LS"] - sc01["LS_Days"]
).abs().max()

max_lf_error = (
    sc01["LF"] - sc01["LF_Days"]
).abs().max()

max_float_error = (
    sc01["Total_Float"] - sc01["Total_Float_Days"]
).abs().max()

assert np.isclose(max_es_error, 0, atol=1e-6)
assert np.isclose(max_ef_error, 0, atol=1e-6)
assert np.isclose(max_ls_error, 0, atol=1e-6)
assert np.isclose(max_lf_error, 0, atol=1e-6)
assert np.isclose(max_float_error, 0, atol=1e-6)

critical_reference = set(
    df_baseline.loc[
        df_baseline["Is_Critical"],
        "ID"
    ]
)

critical_sc01 = set(
    df_scenario_cpm.loc[
        df_scenario_cpm["Scenario_ID"].eq("SC01")
        & df_scenario_cpm["Is_Critical"],
        "ID"
    ]
)

assert critical_reference == critical_sc01

print("CORRECTED CPM VALIDATION")
print(f"Maximum ES error: {max_es_error:.6f}")
print(f"Maximum EF error: {max_ef_error:.6f}")
print(f"Maximum LS error: {max_ls_error:.6f}")
print(f"Maximum LF error: {max_lf_error:.6f}")
print(f"Maximum float error: {max_float_error:.6f}")
print("Critical activities match:", sorted(critical_sc01))

print("\nSCENARIO SUMMARY")
display(df_scenario_summary)

print("\nCRITICAL ACTIVITIES BY SCENARIO")
display(
    df_scenario_cpm[
        df_scenario_cpm["Is_Critical"]
    ][
        [
            "Scenario_ID",
            "ID",
            "Duration_Simulated_Days",
            "ES",
            "EF",
            "LS",
            "LF",
            "Total_Float"
        ]
    ].sort_values(["Scenario_ID", "ID"])
)

| Indicador | SC01: Suministro | SC02: Duración | SC03: Combinado |
| :--- | :--- | :--- | :--- |
| **Duración del proyecto** | 299 días | 306 días | 306 días |
| **Variación vs. baseline** | 0 días | +7 días | +7 días |
| **Actividades críticas** | 3 | 3 | 3 |
| **Cadena crítica** | 12 → 18 → 20 | 12 → 18 → 20 | 12 → 18 → 20 |

In [15]:
# ============================================================
# 09.05 — SCENARIO COMPARISON
# ============================================================

# 1. Preparar referencia de línea base
baseline_ref = df_baseline[
    [
        "ID",
        "ES_Days",
        "EF_Days",
        "Total_Float_Days",
        "Is_Critical"
    ]
].copy()

baseline_ref = baseline_ref.rename(columns={
    "ES_Days": "ES_Baseline",
    "EF_Days": "EF_Baseline",
    "Total_Float_Days": "Float_Baseline",
    "Is_Critical": "Critical_Baseline"
})

# 2. Comparar cada actividad frente a la línea base
df_scenario_comparison = (
    df_scenario_cpm
    .merge(
        baseline_ref,
        on="ID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        df_scenario_durations[
            ["Scenario_ID", "ID", "Activity_Delay_Applied"]
        ],
        on=["Scenario_ID", "ID"],
        how="left",
        validate="one_to_one"
    )
)

# 3. Calcular variaciones
df_scenario_comparison["ES_Variance_Days"] = (
    df_scenario_comparison["ES"] -
    df_scenario_comparison["ES_Baseline"]
)

df_scenario_comparison["EF_Variance_Days"] = (
    df_scenario_comparison["EF"] -
    df_scenario_comparison["EF_Baseline"]
)

df_scenario_comparison["Float_Consumed_Days"] = (
    df_scenario_comparison["Float_Baseline"] -
    df_scenario_comparison["Total_Float"]
)

df_scenario_comparison["Critical_Status_Changed"] = (
    df_scenario_comparison["Is_Critical"] !=
    df_scenario_comparison["Critical_Baseline"]
)

df_scenario_comparison["Schedule_Changed"] = (
    (df_scenario_comparison["ES_Variance_Days"].abs() > 1e-6) |
    (df_scenario_comparison["EF_Variance_Days"].abs() > 1e-6)
)

# 4. Construir resumen ejecutivo por escenario
comparison_summary = []

for scenario_id in sorted(
    df_scenario_comparison["Scenario_ID"].unique()
):

    df_s = df_scenario_comparison[
        df_scenario_comparison["Scenario_ID"] == scenario_id
    ]

    scenario_info = df_scenario_summary[
        df_scenario_summary["Scenario_ID"] == scenario_id
    ].iloc[0]

    supply_delay = df_scenario_supply.loc[
        df_scenario_supply["Scenario_ID"] == scenario_id,
        "Supply_Delay_Days"
    ].sum()

    critical_ids = sorted(
        df_s.loc[df_s["Is_Critical"], "ID"].tolist()
    )

    new_critical = sorted(
        df_s.loc[
            df_s["Is_Critical"] &
            ~df_s["Critical_Baseline"],
            "ID"
        ].tolist()
    )

    no_longer_critical = sorted(
        df_s.loc[
            ~df_s["Is_Critical"] &
            df_s["Critical_Baseline"],
            "ID"
        ].tolist()
    )

    comparison_summary.append({
        "Scenario_ID": scenario_id,
        "Project_Duration_Days": scenario_info[
            "Project_Duration_Days"
        ],
        "Project_Delay_Days": (
            scenario_info["Project_Duration_Days"] -
            BASELINE_DURATION
        ),
        "Activity_Delay_Days": df_s[
            "Activity_Delay_Applied"
        ].sum(),
        "Supply_Delay_Days": supply_delay,
        "Activities_With_Date_Changes": int(
            df_s["Schedule_Changed"].sum()
        ),
        "Activities_With_Float_Consumption": int(
            (df_s["Float_Consumed_Days"] > 1e-6).sum()
        ),
        "Critical_Activities": len(critical_ids),
        "New_Critical_Activities": new_critical,
        "Activities_No_Longer_Critical": no_longer_critical,
        "Critical_Activity_IDs": critical_ids
    })

df_scenario_comparison_summary = pd.DataFrame(
    comparison_summary
)

# 5. Validaciones de coherencia
assert len(df_scenario_comparison) == 135 * 3

# SC01 no debe modificar el calendario
sc01_check = df_scenario_comparison[
    df_scenario_comparison["Scenario_ID"] == "SC01"
]

assert not sc01_check["Schedule_Changed"].any()

# SC02 y SC03 deben desplazar el final de la actividad 20 en 7 días
for scenario_id in ["SC02", "SC03"]:

    activity_20 = df_scenario_comparison[
        (df_scenario_comparison["Scenario_ID"] == scenario_id) &
        (df_scenario_comparison["ID"] == 20)
    ]

    assert np.isclose(
        activity_20["EF_Variance_Days"].iloc[0],
        7.0
    )

# La cadena crítica debe mantenerse
expected_critical_chain = {12, 18, 20}

for scenario_id in ["SC01", "SC02", "SC03"]:

    actual_critical = set(
        df_scenario_comparison.loc[
            (df_scenario_comparison["Scenario_ID"] == scenario_id) &
            df_scenario_comparison["Is_Critical"],
            "ID"
        ]
    )

    assert actual_critical == expected_critical_chain

print("SCENARIO COMPARISON VALIDATED")

print("\nEXECUTIVE SCENARIO COMPARISON")
display(df_scenario_comparison_summary)

print("\nACTIVITIES WITH SCHEDULE CHANGES")
display(
    df_scenario_comparison[
        df_scenario_comparison["Schedule_Changed"]
    ][
        [
            "Scenario_ID",
            "ID",
            "ES_Baseline",
            "ES",
            "ES_Variance_Days",
            "EF_Baseline",
            "EF",
            "EF_Variance_Days",
            "Float_Baseline",
            "Total_Float",
            "Float_Consumed_Days"
        ]
    ].sort_values(["Scenario_ID", "ID"])
)

In [16]:
# ============================================================
# 09.06 — EXPORT SCENARIO SIMULATION RESULTS
# ============================================================

scenario_excel = OUTPUT_DIR / "Scenario_Simulation_Results.xlsx"
scenario_csv = OUTPUT_DIR / "Scenario_Summary.csv"

# 1. Exportar Excel
with pd.ExcelWriter(
    scenario_excel,
    engine="openpyxl"
) as writer:

    df_scenario_comparison_summary.to_excel(
        writer,
        sheet_name="Scenario Summary",
        index=False
    )

    df_scenario_comparison.to_excel(
        writer,
        sheet_name="Activity Comparison",
        index=False
    )

    df_scenario_cpm.to_excel(
        writer,
        sheet_name="Scenario CPM",
        index=False
    )

    df_scenario_supply.to_excel(
        writer,
        sheet_name="Supply Impacts",
        index=False
    )

# 2. Exportar resumen CSV
df_scenario_comparison_summary.to_csv(
    scenario_csv,
    index=False,
    encoding="utf-8-sig"
)

# 3. Verificar archivos y contenido
xls_check = pd.ExcelFile(scenario_excel)

assert xls_check.sheet_names == [
    "Scenario Summary",
    "Activity Comparison",
    "Scenario CPM",
    "Supply Impacts"
]

summary_check = pd.read_excel(
    scenario_excel,
    sheet_name="Scenario Summary"
)

comparison_check = pd.read_excel(
    scenario_excel,
    sheet_name="Activity Comparison"
)

cpm_check = pd.read_excel(
    scenario_excel,
    sheet_name="Scenario CPM"
)

supply_check = pd.read_excel(
    scenario_excel,
    sheet_name="Supply Impacts"
)

assert len(summary_check) == 3
assert len(comparison_check) == 405
assert len(cpm_check) == 405
assert len(supply_check) == 2

assert scenario_csv.exists()
assert scenario_excel.exists()

print("NB_09 COMPLETED SUCCESSFULLY")

print("\nEXCEL SHEETS:")
print(xls_check.sheet_names)

print("\nEXPORTED RECORDS:")
print(f"Scenario summary: {len(summary_check)}")
print(f"Activity comparison: {len(comparison_check)}")
print(f"Scenario CPM: {len(cpm_check)}")
print(f"Supply impacts: {len(supply_check)}")

print("\nFILES:")
print(scenario_excel)
print(scenario_csv)